# From Linear Regression to an MLP in PyTorch

We climb one step at a time; each step changes **one** thing:

| Step | Model | Gradients | Data per update | Update |
|---|---|---|---|---|
| 1 | $wx+b$ | by hand (Ch. 4) | all $N$ points | manual |
| 2 | $wx+b$ | **autograd** | all $N$ points | manual |
| 3 | `nn.Linear` | autograd | **mini-batch** (`DataLoader`) | **optimizer** |
| 4 | **MLP** (`nn.Module`) | autograd | mini-batch | optimizer |

Step 4 changes only the model: the training loop from step 3 stays the same.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from IPython.display import clear_output

## 1. Tensors

A tensor is a numpy array that can (1) live on a GPU and (2) remember how it was computed, so that derivatives can be taken automatically.
Comparison with numpy: https://github.com/torch/torch7/wiki/Torch-for-Numpy-users

In [ ]:
# Create tensors; the dtype matters (integers cannot have gradients)
a = torch.tensor(5, dtype=torch.long)
b = torch.tensor(5, dtype=torch.int32)
b.dtype

In [ ]:
c = torch.ones(6)          # default dtype is float32
print(c, c.dtype)

In [ ]:
# Shape
c.size()                   # same as c.shape

In [ ]:
# Broadcasting: the scalar 1 is added to every element
d = c + 1
print(d)

In [ ]:
# Change type: .float() returns a NEW tensor, b itself is unchanged
print(b.float().dtype, b.dtype)
b = b.float()              # assign to keep the result
print(b.dtype)

In [ ]:
# Reshape tensors (view shares memory with c; reshape may copy)
c.view(2, 3)

In [ ]:
# Squeeze/Unsqueeze: add or remove axes of length 1
print(c.unsqueeze(dim=0).shape, c.unsqueeze(dim=1).shape)

In [ ]:
# Dot product of two 1D tensors vs matrix multiplication
print(d.dot(c))                                   # scalar
print(torch.matmul(d.view(1, -1), c.view(-1, 1))) # (1x6)(6x1) = 1x1 matrix

### In-place operations

Some operations create a new object, others change the given object (in-place). In PyTorch in-place versions end with an underscore:

`x.exp()   # not in-place: returns a new tensor`

`x.exp_()  # in-place: overwrites x`

## 2. Automatic differentiation

A tensor created with `requires_grad=True` records every operation applied to it (the **computational graph**). Calling `.backward()` on a scalar result applies the chain rule backwards through that graph and stores $\partial(\text{result})/\partial(\text{tensor})$ in `tensor.grad`.

Useful attributes: `tensor.data` (the values), `tensor.grad` (the derivative), `tensor.grad_fn` (the operation that produced the tensor).

In [ ]:
a = torch.rand(1, requires_grad=True)
b = torch.sin(a)
print(b.grad_fn)          # b remembers that it came from sin

Use `b.backward()` and compare the derivative `a.grad` with the exact answer $\cos a$.

In [ ]:
b.backward()
print(a.grad, torch.cos(a.data))

### Detaching variables

`.detach()` returns a copy through which backpropagation doesn't pass. If you store batch losses during optimization, store `loss.item()` or `loss.detach()`: otherwise the computational graph of every batch is kept in memory as well.

In [ ]:
x = torch.randn(4, requires_grad=True)
y0 = torch.randn(4, requires_grad=True)
y = y0.detach()            # y has the values of y0 but is cut from the graph
z = x * y + 10
p = z.sum()
p.backward()
print('x:', x.data)
print('y:', y.data)
print('dp / dx:', x.grad)  # = y
print('dp / dy:', y.grad)  # None: y is not tracked

### Gradients accumulate: reset them

`.backward()` **adds** to `.grad`. Before computing a new gradient, set the old one to zero.

In [ ]:
x = torch.tensor([1, 1, 1, 1.], requires_grad=True)
p = (x ** 2).sum()
p.backward()
print('dp / dx:', x.grad)           # 2x = 2

p = (1 / x).sum()
p.backward()
print('without reset:', x.grad)     # 2 + (-1/x^2) = 1: the two gradients were added!

x.grad.zero_()                      # reset
p = (1 / x).sum()
p.backward()
print('with reset:', x.grad)        # -1/x^2 = -1

### Leaf vs non-leaf variables

Tensors created by you are **leaves**; tensors computed from them are not. By default `.grad` is kept only for leaves (the parameters we want to update).

In [ ]:
x = torch.randn(4, requires_grad=True)  # leaf variable
y = x + 1                               # not a leaf variable
p = y.sum()                             # not a leaf variable
p.backward()
print('x.grad:', x.grad)
print('y.grad:', y.grad)                # None (PyTorch also prints a warning)

### Gradients for non-leaf variables: `retain_grad()`

In [ ]:
x = torch.randn(4, requires_grad=True)   # leaf variable
z = torch.randn(4, requires_grad=True)   # leaf variable
w = z * 2      # not a leaf variable
y = x * w + 1
p = y.sum()

w.retain_grad()                          # ask PyTorch to keep dp/dw

p.backward()
print('dp / dx:', x.grad)                # = w
print('dp / dz:', z.grad)                # = 2x  (chain rule through w)
print('dp / dw:', w.grad)                # = x

## 3. Linear regression: autograd = the formula from Chapter 4

For $\hat y = wx+b$ and $L = \frac1N\sum_i(\hat y_i - y_i)^2$:
$$\frac{\partial L}{\partial w} = \frac2N\sum_i(\hat y_i-y_i)\,x_i,\qquad \frac{\partial L}{\partial b} = \frac2N\sum_i(\hat y_i-y_i).$$

In [ ]:
torch.manual_seed(0)
x = torch.linspace(1, 5, 20)
y = 2*x + 1 + 5*torch.rand(len(x))     # true line 2x + 1 plus noise with mean 2.5
plt.scatter(x, y)

### Q1 (1 pt). Initialize the weight and bias as tensors with gradients (see the example with `sin(a)`).

In [ ]:
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)


### Q2 (1 pt). Check autograd against the formula.
Compute the loss once, call `loss.backward()`, compare `w.grad`, `b.grad` with the formulas above, then zero the gradients.

In [ ]:
y_p = w*x + b
loss = torch.mean((y_p - y)**2)
loss.backward()
with torch.no_grad():
    print('autograd:', w.grad.item(), b.grad.item())
    print('formula :', (2*torch.mean((y_p - y)*x)).item(), (2*torch.mean(y_p - y)).item())
w.grad.zero_(); b.grad.zero_()          # clean up before training


### Q3 (2 pt). Gradient descent with automatically computed gradients.

Updates of parameters must not be recorded in the graph, so we do them inside `with torch.no_grad():` (older code writes `w.data = ...`; this is discouraged because it bypasses autograd's checks).

In [ ]:
alpha = 0.05   # learning rate; for this x, gradient descent diverges above about 0.088
for i in range(300):   # number of epochs = number of updates here (full batch)
    # calculate prediction y_p = w*x+b:
    y_p = w*x + b

    # calculate loss mean((y_p-y)**2):
    loss = torch.mean((y_p - y)**2)

    # run loss.backward() to calculate the derivatives
    loss.backward()

    # update w and b using the learning rate alpha and the gradients, then zero the gradients
    with torch.no_grad():
        w -= alpha * w.grad
        b -= alpha * b.grad
        w.grad.zero_()
        b.grad.zero_()

    # visualisation:
    if (i + 1) % 50 == 0:
        clear_output(True)
        plt.figure(figsize=(8, 6))
        plt.scatter(x, y, label="data")
        plt.plot(x, y_p.detach().numpy(), color="orange", linewidth=3, label="predictions")
        plt.title(f"epoch {i+1}, loss {loss.item():.3f}")
        plt.legend()
        plt.show()

print(w.item(), b.item())

**Question.** $w$ settles quickly, but $b$ creeps slowly towards its final value. Why? *(Hint: $x\in[1,5]$ is not centered, so the loss surface is a long narrow valley. What happens if you use $x - \bar x$ instead?)*

## 4. Mini-batches, `nn.Linear` and an optimizer

Instead of all $N$ points, each update uses a random **mini-batch** of $B$ points (Notes 5.2). One pass over the data is an **epoch**; one update is an **iteration**, so an epoch has $\lceil N/B\rceil$ iterations.

PyTorch splits this into three pieces:
* `Dataset`: how to get the $i$-th example;
* `DataLoader`: groups examples into batches and **shuffles** them every epoch;
* `optimizer`: performs the update `w -= lr * w.grad` (or a smarter one) for all parameters.

In [ ]:
class MyData(Dataset):
    def __init__(self, x, y):
        super().__init__()
        self.x = x          # tensors of shape (N, 1)
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]


data = MyData(x.view(-1, 1), y.view(-1, 1))   # nn layers expect shape (batch, features)
print(data[2])

# shuffle=True: a new random split into batches every epoch
data_loader = DataLoader(data, batch_size=5, shuffle=True)
for x_batch, y_batch in data_loader:
    print(x_batch.shape, y_batch.shape)
    break

`nn.Linear(1, 1)` is exactly $wx+b$: linear regression is a neural network with no hidden layer.

In [ ]:
torch.manual_seed(0)
model = nn.Linear(1, 1)
print(list(model.parameters()))       # weight w and bias b, created with requires_grad=True

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.02)

### Q4 (2 pt). Write the mini-batch training loop (200 epochs).
Per batch: prediction, loss, `backward`, `optimizer.step()`, `optimizer.zero_grad()`. Store `loss.item()` in `hist`.

In [ ]:
hist = []
for epoch in range(200):
    for x_b, y_b in data_loader:
        y_pred = model(x_b)            # model(x) calls model.forward(x)
        loss = criterion(y_pred, y_b)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        hist.append(loss.item())       # .item(): a plain number, no graph kept

plt.plot(hist)
plt.xlabel('iteration'); plt.ylabel('batch loss')
print(model.weight.item(), model.bias.item())

**Question.** Why is the loss curve noisy even at the end, while in Q3 it decreased smoothly?

## 5. A two-layer perceptron

Now a target that no line can fit: $y = |x|$. We change **only the model**; the loop from Q4 is reused.

$$a(x) = W_2\,\sigma(W_1 x + b_1) + b_2$$

In [ ]:
x2 = torch.linspace(-1, 1, 100).view(-1, 1)
y2 = torch.abs(x2)
plt.plot(x2, y2)

data2 = MyData(x2, y2)
data_loader2 = DataLoader(data2, batch_size=10, shuffle=True)

In [ ]:
class MyModel(nn.Module):
    def __init__(self, hidden_size, activation=nn.Sigmoid()):
        super().__init__()
        self.fc1 = nn.Linear(1, hidden_size)     # W1 x + b1
        self.activ = activation                  # sigma, applied elementwise
        self.fc2 = nn.Linear(hidden_size, 1)     # W2 h + b2

    def forward(self, x_batch):
        o = self.fc1(x_batch)
        o = self.activ(o)
        o = self.fc2(o)                          # keep the output of the last layer
        return o

In [ ]:
def train(model, loader, epochs=100, lr=0.01):
    # the same loop as in Q4, with Adam instead of plain SGD
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    hist = []
    for epoch in range(epochs):
        for x_b, y_b in loader:
            loss = criterion(model(x_b), y_b)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            hist.append(loss.item())
    return hist


def show(model, title):
    with torch.no_grad():                # no graph needed for plotting
        y_pred = model(x2)
    plt.scatter(x2, y2, s=8, label='|x|')
    plt.plot(x2, y_pred, color='orange', linewidth=3, label='model')
    plt.title(title); plt.legend(); plt.show()

### Q5 (2 pt). Train three models and compare the fits:
1. `nn.Linear(1, 1)` (no hidden layer);
2. `MyModel(5)` with the sigmoid;
3. `MyModel(8, nn.ReLU())`.

In [ ]:
torch.manual_seed(0)
for name, m in [('linear', nn.Linear(1, 1)),
                ('MLP, 5 sigmoid units', MyModel(5)),
                ('MLP, 8 ReLU units', MyModel(8, nn.ReLU()))]:
    hist = train(m, data_loader2, epochs=200, lr=0.01)
    show(m, f'{name}: final batch loss {hist[-1]:.4f}')


### Q6 (2 pt). Exact representation.
Show that $|x| = \mathrm{ReLU}(x) + \mathrm{ReLU}(-x)$, and write down $W_1, b_1, W_2, b_2$ of a network with **2** ReLU units that equals $|x|$ exactly.
Then train `MyModel(2, nn.ReLU())` from several random seeds. Does training always find this solution? What goes wrong when it does not? *(Hint: look at units whose pre-activation $W_1x+b_1$ is negative for every $x\in[-1,1]$.)*

In [ ]:
# W1 = [[1], [-1]], b1 = [0, 0], W2 = [[1, 1]], b2 = [0]
exact = MyModel(2, nn.ReLU())
with torch.no_grad():
    exact.fc1.weight.copy_(torch.tensor([[1.], [-1.]]))
    exact.fc1.bias.zero_()
    exact.fc2.weight.copy_(torch.tensor([[1., 1.]]))
    exact.fc2.bias.zero_()
show(exact, 'hand-set weights: exact')

for seed in range(4):
    torch.manual_seed(seed)
    m = MyModel(2, nn.ReLU())
    hist = train(m, data_loader2, epochs=200, lr=0.01)
    show(m, f'seed {seed}: final batch loss {hist[-1]:.4f}')
# Some seeds may give a "dead" unit (ReLU output 0 on the whole interval -> zero gradient),
# and the network is stuck with a single kink.
